In [1]:
FAISS_PATH='../vectorstore/faiss_index'

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "llama3.2:3b",
    temperature=0.0
)
response = llm.invoke("Hola, funcionas en local?")
print(response.content)

¡Hola! Me alegra que hayas intentado contactarme. Sin embargo, como soy una inteligencia artificial, no tengo una ubicación física específica y puedo acceder a internet desde cualquier lugar.

Puedo proporcionarte información y responder a tus preguntas de manera remota, siempre y cuando tengas acceso a una conexión a Internet. ¿En qué puedo ayudarte hoy?


In [3]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

# cargamos la knowledge
folder_loader = DirectoryLoader(
    "../knowledge/production_simulation",
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader
)
docs_md = folder_loader.load()

# cargamos los json
def json_metadata(record: dict, metadata: dict):
    metadata["traceId"] = record.get("traceId", "NONE")
    metadata["service"] = record.get("service", "NONE")
    metadata["level"] = record.get("level", "NONE")

    return metadata

json_arguments = {
    "jq_schema": '.[] | select(has("error_message"))',
    "content_key": "error_message",
    "metadata_func": json_metadata
}

json_loader = DirectoryLoader(
    "../datasets",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs=json_arguments
)
docs_json = json_loader.load()

print(f"Archivos .md cargados: {len(docs_md)}")
print(f"Archivos JSON cargados: {len(docs_json)}")

/var/folders/1r/npnvwl1n38g5mh09lhx8lgdm0000gn/T/ipykernel_36230/4234408129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredMarkdownLoader


Archivos .md cargados: 4
Archivos JSON cargados: 2279


In [4]:
docs = docs_md + docs_json

# ahora con los datos completos los pasamos por el textsplitter y preparamos el vectorstore
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

headers_to_split_on = [
    ("#", "titulo"),
    ("##", "seccion"),
    ("###", "subseccion"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
)

embedding = OllamaEmbeddings(
    model = "nomic-embed-text"
)

chunks = text_splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")
print(f"Chunks vacíos: {sum(1 for c in chunks if not c.page_content.strip())}")
print(f"Chunk más largo: {max(len(c.page_content) for c in chunks)} chars")
print(f"Ejemplo chunk 0:\n{chunks[0].page_content[:200]}")

# Elimina chunks vacíos antes de pasarlos
chunks = [c for c in chunks if c.page_content.strip()]

vectorstore = FAISS.from_documents(chunks, embedding)

vectorstore.save_local(FAISS_PATH)

retriever = vectorstore.as_retriever()

Total chunks: 2334
Chunks vacíos: 0
Chunk más largo: 797 chars
Ejemplo chunk 0:
Formato de Logs y Referencia de Campos — RCA Platform

Este documento describe el formato exacto de los logs estructurados del sistema, el significado de cada campo y las reglas de interpretación para


In [5]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

template = """
    Eres un Ingeniero SRE (Site Reliability Engineer) y Experto Forense de Nivel 3. 
    Tu especialidad es diagnosticar fallos en cascada en una arquitectura de microservicios Spring Boot (api-pedidos, api-inventario, api-autenticacion).

    Has recibido una alerta de nuestro modelo de Machine Learning (Isolation Forest) indicando que el siguiente bloque de logs es una ANOMALÍA CRÍTICA.

    REGLAS ESTRICTAS:
    1. NO inventes información. Utiliza ÚNICAMENTE la documentación técnica y los runbooks proporcionados en el apartado <contexto>.
    2. Si el <contexto> no contiene la respuesta, di explícitamente: "No hay información suficiente en los manuales para determinar la causa raíz."
    3. Debes diferenciar el "paciente cero" (causa raíz) de las víctimas (errores en cascada).

    <contexto>
    {context}
    </contexto>

    FORMATO DE RESPUESTA OBLIGATORIO:
    Responde siempre usando esta estructura en Markdown:

    🚨 **Análisis de Causa Raíz (RCA)**
    * **Microservicio Origen:** [Nombre del servicio que falló primero]
    * **Excepción Principal:** [Tipo de error, ej. NullPointerException, Timeout]
    * **Diagnóstico:** [Explicación técnica de 2 o 3 líneas de por qué ocurrió según el contexto]

    🛠️ **Plan de Mitigación (Runbook)**
    1. [Paso 1 para solucionarlo]
    2. [Paso 2 para solucionarlo]
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    MessagesPlaceholder(variable_name="history"),
    ("human:" "Analiza este log anomalo y dime qué ha pasado: \n\n{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnablePassthrough.assign(
        context=itemgetter("input") | retriever | format_docs
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [6]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# almacenamos el historial segun el usuario
chat_history = ChatMessageHistory()

def obtener_historial_por_session_id(session_id: str):
  return chat_history

RAG = RunnableWithMessageHistory(
  rag_chain,
  obtener_historial_por_session_id,
  input_messages_key="input",
  history_messages_key="history"
)

/opt/homebrew/Caskroom/miniforge/base/envs/rca/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
test_log = """
TRACE_ID: 6a721f115975b8d2d0fd94c37ef6598a

[08:00:00] SERVICIO: api-inventario | EVENTO: SERVER_ERROR | DURACION: 30106ms
EXCEPCION: CannotCreateTransactionException
MENSAJE: could not open jpa entitymanager for transaction

[08:00:05] SERVICIO: api-pedidos | EVENTO: UNHANDLED_ERROR | DURACION: 5185ms
EXCEPCION: ResourceAccessException
MENSAJE: i o error on post request for http api inventario - read timed out
}
"""

# Ejecutamos el bot
respuesta = RAG.invoke(
    {"input": test_log},
    config={"configurable": {"session_id": "prueba_test_01"}}
)
print(respuesta)

🚨 **Análisis de Causa Raíz (RCA)**

* **Microservicio Origen:** [api-inventario]
* **Excepción Principal:** [CannotCreateTransactionException]
* **Diagnóstico:** El servicio api-inventario falló debido a un problema con la conexión a la base de datos, lo que impidió abrir el EntityManager para una transacción. Esto se debe a que el tiempo de espera del pool de conexiones (HikariPool) fue agotado, lo que causó un timeout.

🛠️ **Plan de Mitigación (Runbook)**

1. Verificar y aumentar la cantidad máxima de conexiones en el HikariPool para reducir el tiempo de espera.
2. Investigar y solucionar cualquier problema con la base de datos o la configuración del pool de conexiones que pueda estar causando este error.

Nota: Es importante notar que el servicio api-pedidos está consumiendo el mismo traceId que el servicio api-inventario, lo que sugiere una posible cascada de errores. Sin embargo, en este caso, el error en el servicio api-pedidos es un ResourceAccessException debido a un timeout, l